# Tiny Dreamer Highway — Colab Git + Drive Setup

**Name:** Esteban  
**Course:** CSC 580 AI 2  
**Assignment:** Final Project — Dream the Road  
**AI tools consulted:** GitHub Copilot

This notebook follows the project rule: GitHub is the source of truth for code, and Google Drive stores artifacts such as checkpoints, plots, and videos.

## Runtime flow

1. Mount Google Drive.
2. Clone or pull the latest repository snapshot from GitHub.
3. Install the package and dependencies.
4. Run a small sanity check before training.
5. Save outputs into the Drive-backed `artifacts/` folder.

In [17]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO_URL = 'https://github.com/estmon8u/CSC_580_Final_Project.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
WORKTREE = Path('/content/CSC_580_Final_Project')

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / 'checkpoints', ARTIFACT_ROOT / 'media', ARTIFACT_ROOT / 'logs']:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)

print('Artifact root:', ARTIFACT_ROOT)

print('Repository URL:', REPO_URL)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root: /content/drive/MyDrive/CSC_580_Final_Project
Artifact root: /content/drive/MyDrive/CSC_580_Final_Project/artifacts
Repository URL: https://github.com/estmon8u/CSC_580_Final_Project.git


In [18]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git pull --ff-only origin main
fi

Updating 8641e1c..66b3be7
Fast-forward
 notebooks/01_colab_git_drive_setup.ipynb    |  80 ++++++++++++++++++----
 src/tiny_dreamer_highway/models/__init__.py |   3 +-
 src/tiny_dreamer_highway/models/rssm.py     | 101 ++++++++++++++++++++++++++++
 tests/test_rssm.py                          |  62 +++++++++++++++++
 4 files changed, 233 insertions(+), 13 deletions(-)
 create mode 100644 src/tiny_dreamer_highway/models/rssm.py
 create mode 100644 tests/test_rssm.py


From https://github.com/estmon8u/CSC_580_Final_Project
 * branch            main       -> FETCH_HEAD
   8641e1c..66b3be7  main       -> origin/main


In [19]:
%%bash
set -e
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip --quiet

MARKER=/content/CSC_580_Final_Project/.runtime_restart_required

# Check the numpy version currently active in this runtime's Python
CURRENT_NP=$(python -c "import numpy; print(numpy.__version__)" 2>/dev/null || echo "none")
echo "Runtime numpy: ${CURRENT_NP}"

if [ "${CURRENT_NP}" != "2.0.2" ]; then
  echo "NumPy is ${CURRENT_NP}, not 2.0.2 — force-reinstalling scientific stack from PyPI..."
  # Force download fresh wheels compiled for numpy 2.0 (replaces Colab's pre-built binaries)
  python -m pip install --force-reinstall --quiet "numpy==2.0.2" "scipy==1.14.1"
  echo "restart-required" > "${MARKER}"
  echo ""
  echo ">>> STOP: Restart the Colab runtime now, then rerun from the top. <<<"
else
  echo "NumPy 2.0.2 already active — installing remaining dependencies normally..."
  python -m pip install -e . --quiet
  rm -f "${MARKER}"
  echo "Install complete — no restart required."
fi


Runtime numpy: 2.0.2
NumPy 2.0.2 already active — installing remaining dependencies normally...
Install complete — no restart required.


## Dependency source of truth

This notebook installs the project directly from the repository with `pip install -e .`. All dependency versions come from `pyproject.toml`.

### Why the install cell checks the runtime numpy version

Colab ships its own pre-built `scipy` binary compiled against numpy 1.x. If we simply run `pip install scipy==1.14.1`, pip sees the version already matches and skips the download — leaving the old numpy-1.x-compiled `.so` files in place. Mixing those with numpy 2.0 at import time causes `AttributeError: module 'numpy' has no attribute '_no_nep50_warning'`.

**The fix:** the install cell checks `numpy.__version__` in the *running* Python interpreter (not pip metadata):
- If it is **not** `2.0.2`, it runs `pip install --force-reinstall numpy==2.0.2 scipy==1.14.1` to download fresh PyPI wheels compiled for numpy 2.0, writes a restart marker, and stops.
- If it **is already** `2.0.2`, it runs `pip install -e .` normally (numpy/scipy are already correct), removes the marker, and continues.

**Expected flow on a fresh Colab runtime:**
1. Run install cell → detects numpy 1.x → force-reinstalls → prints STOP message.
2. Restart runtime (*Runtime → Restart runtime*).
3. Re-run all cells from the top → install cell detects numpy 2.0.2 → normal install → no restart needed.
4. Version-check cell passes → continue.


In [20]:
from importlib.metadata import version
from pathlib import Path

restart_marker = Path('/content/CSC_580_Final_Project/.runtime_restart_required')

print('numpy version:', version('numpy'))
print('scipy version:', version('scipy'))
print('gymnasium version:', version('gymnasium'))
print('highway-env version:', version('highway-env'))

if restart_marker.exists():
    raise RuntimeError(
        'The install step changed NumPy or SciPy in this active runtime. '
        'Restart the Colab runtime now, then rerun the notebook from the top. '
        'After restart, the install cell should leave the binary packages unchanged and this check will pass.'
    )

print('\nBinary dependency state looks stable. Safe to continue.')


numpy version: 2.0.2
scipy version: 1.14.1
gymnasium version: 1.2.3
highway-env version: 1.10.2

Binary dependency state looks stable. Safe to continue.


In [21]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.cli import summarize_config

config_path = PROJECT_ROOT / 'examples' / 'base_experiment.yaml'
config = load_experiment_config(config_path)
print(summarize_config(config))

Loaded config for highway-v0 | obs=64x64 | replay_capacity=10000 | sequence_length=8 | batch_size=4


In [22]:
from tiny_dreamer_highway.envs.highway_factory import make_highway_env

env = make_highway_env(config.env)
observation, info = env.reset()
action = env.action_space.sample()
next_observation, reward, terminated, truncated, info = env.step(action)
env.close()

print('Observation shape:', getattr(observation, 'shape', None))
print('Action shape:', getattr(action, 'shape', None))
print('Reward:', reward)
print('Episode ended:', bool(terminated or truncated))

Observation shape: (1, 64, 64)
Action shape: (2,)
Reward: 0.7845891722156991
Episode ended: False


## Replay warm-start smoke test

This cell exercises milestone M2: collect a small batch of random transitions, then verify that replay batch sampling and sequence sampling both work.

In [23]:
from tiny_dreamer_highway.data.collect_random_rollouts import collect_random_transitions
from tiny_dreamer_highway.data.replay_buffer import ReplayBuffer

replay_buffer = ReplayBuffer(capacity=config.replay.capacity)
added = collect_random_transitions(config.env, replay_buffer, steps=32)

batch = replay_buffer.sample_batch(batch_size=config.replay.batch_size)
sequences = replay_buffer.sample_sequences(
    batch_size=config.replay.batch_size,
    sequence_length=config.replay.sequence_length,
)

print('Collected transitions:', added)
print('Replay size:', len(replay_buffer))
print('Batch observation shape:', batch.observations.shape)
print('Batch action shape:', batch.actions.shape)
print('Sequence batch:', len(sequences), 'x', len(sequences[0]))

Collected transitions: 32
Replay size: 32
Batch observation shape: (4, 1, 64, 64)
Batch action shape: (4, 2)
Sequence batch: 4 x 8


## Encoder shape smoke test

This cell starts milestone M3 by pushing a sampled replay batch through the new observation encoder and verifying the latent feature shape.

In [24]:
import torch
from tiny_dreamer_highway.models import ObservationEncoder

observation_tensor = torch.as_tensor(batch.observations)
if observation_tensor.ndim == 3:
    observation_tensor = observation_tensor.unsqueeze(1)

encoder = ObservationEncoder(
    in_channels=observation_tensor.shape[1],
    observation_shape=tuple(observation_tensor.shape[-2:]),
    embedding_dim=256,
)
latent = encoder(observation_tensor)

print('Encoder input shape:', tuple(observation_tensor.shape))
print('Latent embedding shape:', tuple(latent.features.shape))
print('Conv output shape:', encoder.conv_output_shape)

Encoder input shape: (4, 1, 64, 64)
Latent embedding shape: (4, 256)
Conv output shape: (256, 4, 4)


## RSSM shape smoke test

This cell advances milestone M3 by feeding encoder embeddings and replay actions through the RSSM prior and posterior updates and verifying the latent state shapes.

In [26]:
try:
    from tiny_dreamer_highway.models import RecurrentStateSpaceModel
except ImportError:
    from tiny_dreamer_highway.models.rssm import RecurrentStateSpaceModel

action_tensor = torch.as_tensor(batch.actions, dtype=torch.float32)
embedding_tensor = latent.embedding

rssm = RecurrentStateSpaceModel(
    action_dim=action_tensor.shape[-1],
    embedding_dim=embedding_tensor.shape[-1],
    deterministic_dim=128,
    stochastic_dim=32,
)
initial_state = rssm.initial_state(batch_size=action_tensor.shape[0], device=action_tensor.device)
prior_state = rssm.imagine_step(initial_state, action_tensor)
posterior_state = rssm.observe_step(initial_state, action_tensor, embedding_tensor)

print('Action shape:', tuple(action_tensor.shape))
print('Embedding shape:', tuple(embedding_tensor.shape))
print('Prior deterministic shape:', tuple(prior_state.deterministic.shape))
print('Prior stochastic shape:', tuple(prior_state.stochastic.shape))
print('Posterior deterministic shape:', tuple(posterior_state.deterministic.shape))
print('Posterior stochastic shape:', tuple(posterior_state.stochastic.shape))

Action shape: (4, 2)
Embedding shape: (4, 256)
Prior deterministic shape: (4, 128)
Prior stochastic shape: (4, 32)
Posterior deterministic shape: (4, 128)
Posterior stochastic shape: (4, 32)


## Next step

After the RSSM smoke test succeeds, implement the decoder and reward prediction heads so the world model can reconstruct frames and predict rewards from latent features.